# 10_Model_Evaluation.ipynb
Evaluate the trained multimodal model using common classification metrics.

In [ ]:
!pip -q install torch pandas scikit-learn matplotlib

import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

from google.colab import files

print("Upload cnn_features.csv")
u=files.upload()
cnn=pd.read_csv(list(u.keys())[0])

print("Upload transformer_features.csv")
u=files.upload()
tr=pd.read_csv(list(u.keys())[0])

print("Upload multimodal_model.pth")
u=files.upload()
model_path=list(u.keys())[0]

def clean_ids(s):
    return (s.astype(str)
              .str.replace("_Sentinel-2","",regex=False)
              .str.replace("_Landsat","",regex=False)
              .str.strip())

cnn["event_id"]=clean_ids(cnn["event_id"])
tr["event_id"]=clean_ids(tr["event_id"])

cnn=cnn.rename(columns={c:f"cnn_{c}" for c in cnn.columns if c!="event_id"})
tr=tr.rename(columns={c:f"tr_{c}" for c in tr.columns if c!="event_id"})

df=cnn.merge(tr,on="event_id",how="inner")

labels=df["event_id"].apply(lambda x:0 if "FLASH" in x.upper() else 1)

cnn_cols=[c for c in df.columns if c.startswith("cnn_")]
tr_cols=[c for c in df.columns if c.startswith("tr_")]

Xcnn=df[cnn_cols].values
Xtr=df[tr_cols].values
y=labels.values

Xcnn_train,Xcnn_test,Xtr_train,Xtr_test,y_train,y_test=train_test_split(
    Xcnn,Xtr,y,test_size=0.2,random_state=42,stratify=y
)

class FusionModel(nn.Module):
    def __init__(self,dim=512):
        super().__init__()
        self.attn=nn.MultiheadAttention(dim,8,batch_first=True)
        self.head=nn.Sequential(
            nn.Linear(dim,256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,2)
        )
    def forward(self,cnn,tr):
        out,_=self.attn(cnn.unsqueeze(1),tr.unsqueeze(1),tr.unsqueeze(1))
        return self.head(out.squeeze(1))

device="cuda" if torch.cuda.is_available() else "cpu"

model=FusionModel().to(device)
model.load_state_dict(torch.load(model_path,map_location=device))
model.eval()

with torch.no_grad():
    pred=model(
        torch.tensor(Xcnn_test,dtype=torch.float32).to(device),
        torch.tensor(Xtr_test,dtype=torch.float32).to(device)
    ).argmax(1).cpu().numpy()

acc=accuracy_score(y_test,pred)
pre=precision_score(y_test,pred,zero_division=0)
rec=recall_score(y_test,pred,zero_division=0)
f1=f1_score(y_test,pred,zero_division=0)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {pre:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nClassification Report")
print(classification_report(y_test,pred,target_names=["FLASH","HEAT"],zero_division=0))

cm=confusion_matrix(y_test,pred)
disp=ConfusionMatrixDisplay(cm,display_labels=["FLASH","HEAT"])
disp.plot()

plt.title("Confusion Matrix")
plt.savefig("confusion_matrix.png",dpi=300)
plt.show()

files.download("confusion_matrix.png")
